In [1]:
import math
from pathlib import Path
from tqdm.notebook import tqdm
from moviepy import VideoFileClip, AudioFileClip 

# Extensions map
AUDIO_EXTS = {".m4a"}

def get_unique_output_folder(base_path: Path, folder_name: str) -> Path:
    """Creates a directory, appending _v1, _v2 if it exists to avoid overwrites."""
    out_path = base_path / folder_name
    if not out_path.exists():
        out_path.mkdir(parents=True)
        return out_path
    
    counter = 1
    while True:
        new_path = base_path / f"{folder_name}_v{counter}"
        if not new_path.exists():
            new_path.mkdir(parents=True)
            return new_path
        counter += 1

In [2]:
AUDIO_EXTS = {".m4a", ".wav"}

def segment_media(
    file_path: str | Path, 
    segment_duration: float, 
    output_base_dir: str | Path = None,
    drop_last: bool = True
):
    """
    Segments media into chunks.
    - Audio inputs (.m4a) -> Saved as .wav (PCM_16)
    - Video inputs (.mp4) -> Saved as .mp4 (Mute)
    """
    input_path = Path(file_path)
    if not input_path.exists():
        print(f" File not found: {input_path}")
        return

    # 1. Setup Paths
    is_audio = input_path.suffix.lower() in AUDIO_EXTS
    stem = input_path.stem
    output_base_dir = Path(output_base_dir) if output_base_dir else input_path.parent

    # Folder naming convention: e.g., Video1_chunks_2s
    folder_name = f"{stem}_chunks_{int(segment_duration)}s"
    save_dir = output_base_dir / folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    # 2. Load Clip
    try:
        if is_audio:
            clip = AudioFileClip(str(input_path))
        else:
            clip = VideoFileClip(str(input_path))
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    duration = clip.duration
    
    # 3. Calculate Chunks
    if drop_last:
        num_chunks = math.floor(duration / segment_duration)
        drop_msg = "(Dropping last partial chunk)"
    else:
        num_chunks = math.ceil(duration / segment_duration)
        drop_msg = "(Keeping last partial chunk)"
        
    pad_len = max(3, len(str(num_chunks)))

    print(f"--- Processing: {input_path.name} ---")
    print(f"Type: {'Audio' if is_audio else 'Video'}")
    print(f"Duration: {duration:.2f}s | Segment: {segment_duration}s | Count: {num_chunks}")
    print(f"Mode: {drop_msg} | Output: {save_dir}")

    # 4. Processing Loop
    for i in tqdm(range(num_chunks), desc=f"Segmenting {stem}"):
        start_t = i * segment_duration
        end_t = min((i + 1) * segment_duration, duration)
        
        # Determine Output Filename
        if is_audio:
            # FORCE .wav extension for audio segments
            chunk_name = f"{stem}_part_{i+1:0{pad_len}d}.wav"
        else:
            # Keep original extension for video
            chunk_name = f"{stem}_part_{i+1:0{pad_len}d}{input_path.suffix}"
            
        chunk_path = save_dir / chunk_name
        
        try:
            
            subclip = clip.subclipped(start_t, end_t)
            
            if is_audio:
                # Write as standard 16-bit PCM WAV (compatible with everything)
                subclip.write_audiofile(
                    str(chunk_path), 
                    codec='pcm_s16le', 
                    logger=None
                )
            else:
                # Write video WITHOUT audio 
                subclip.write_videofile(
                    str(chunk_path),
                    codec="libx264",
                    audio=False,      
                    preset="ultrafast", 
                    threads=4,
                    logger=None
                )
        except Exception as e:
            print(f"Failed on chunk {i+1}: {e}")
        
    clip.close()
    print(f"Finished. Saved to: {save_dir}\n")

In [3]:
# --- Configuration ---
VIDEO_ID = 1
SEGMENT_SEC = 10.0

# Define specific paths (Modify these two lines)
INPUT_ROOT = Path(r"/home/amin/Research/Representation/Movie/data/Setareh/Data") 
TARGET_FILE = INPUT_ROOT / f"Video{VIDEO_ID}/Video{VIDEO_ID}.mp4"

# --- Run ---
segment_media(
    file_path=TARGET_FILE, 
    segment_duration=SEGMENT_SEC
)

--- Processing: Video1.mp4 ---
Type: Video
Duration: 212.00s | Segment: 10.0s | Count: 21
Mode: (Dropping last partial chunk) | Output: /home/amin/Research/Representation/Movie/data/Setareh/Data/Video1/Video1_chunks_10s


Segmenting Video1:   0%|          | 0/21 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
SEGMENT_SEC_LIST = [5.0 , 10.0]
INPUT_ROOT = Path(r"/home/amin/Research/Representation/Movie/data/Setareh/Data") 
for MEDIA_ID in range(18):
    TARGET_VIDEO = INPUT_ROOT / f"Video{MEDIA_ID+1}/Video{MEDIA_ID+1}.mp4"
    TARGET_AUDIO = INPUT_ROOT / f"Audio{MEDIA_ID+1}/Audio{MEDIA_ID+1}.m4a"
    print(f"Processing Segment {MEDIA_ID+1}")
    for segment_duration in SEGMENT_SEC_LIST:
        segment_media(TARGET_VIDEO, segment_duration=segment_duration)
        print(f"{TARGET_VIDEO} DONE")
        segment_media(TARGET_AUDIO, segment_duration=segment_duration)
        print(f"{TARGET_AUDIO} DONE")

Processing Segment 1
--- Processing: Video1.mp4 ---
Type: Video
Duration: 212.00s | Segment: 5.0s | Count: 42
Mode: (Dropping last partial chunk) | Output: /home/amin/Research/Representation/Movie/data/Setareh/Data/Video1/Video1_chunks_5s


Segmenting Video1:   0%|          | 0/42 [00:00<?, ?it/s]

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/Data/Video1/Video1_chunks_5s

/home/amin/Research/Representation/Movie/data/Setareh/Data/Video1/Video1.mp4 DONE
--- Processing: Audio1.m4a ---
Type: Audio
Duration: 212.04s | Segment: 5.0s | Count: 42
Mode: (Dropping last partial chunk) | Output: /home/amin/Research/Representation/Movie/data/Setareh/Data/Audio1/Audio1_chunks_5s


Segmenting Audio1:   0%|          | 0/42 [00:00<?, ?it/s]

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/Data/Audio1/Audio1_chunks_5s

/home/amin/Research/Representation/Movie/data/Setareh/Data/Audio1/Audio1.m4a DONE
--- Processing: Video1.mp4 ---
Type: Video
Duration: 212.00s | Segment: 10.0s | Count: 21
Mode: (Dropping last partial chunk) | Output: /home/amin/Research/Representation/Movie/data/Setareh/Data/Video1/Video1_chunks_10s


...

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/Data/Audio18/Audio18_chunks_5s

/home/amin/Research/Representation/Movie/data/Setareh/Data/Audio18/Audio18.m4a DONE
--- Processing: Video18.mp4 ---
Type: Video
Duration: 82.08s | Segment: 10.0s | Count: 8
Mode: (Dropping last partial chunk) | Output: /home/amin/Research/Representation/Movie/data/Setareh/Data/Video18/Video18_chunks_10s


Segmenting Video18:   0%|          | 0/8 [00:00<?, ?it/s]

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/Data/Video18/Video18_chunks_10s

/home/amin/Research/Representation/Movie/data/Setareh/Data/Video18/Video18.mp4 DONE
--- Processing: Audio18.m4a ---
Type: Audio
Duration: 82.29s | Segment: 10.0s | Count: 8
Mode: (Dropping last partial chunk) | Output: /home/amin/Research/Representation/Movie/data/Setareh/Data/Audio18/Audio18_chunks_10s


Segmenting Audio18:   0%|          | 0/8 [00:00<?, ?it/s]

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/Data/Audio18/Audio18_chunks_10s

/home/amin/Research/Representation/Movie/data/Setareh/Data/Audio18/Audio18.m4a DONE


In [2]:
import math
from pathlib import Path
from tqdm.notebook import tqdm
from moviepy import VideoFileClip, AudioFileClip 

# Re-define locally to add trim_end_sec logic specifically for the full runs
AUDIO_EXTS = {".m4a", ".wav"}

def segment_media_trimmed(
    file_path: str | Path, 
    segment_duration: float, 
    output_base_dir: str | Path = None,
    drop_last: bool = True,
    trim_end_sec: float = 5.0  # <--- Automatically drops the last 5s
):
    input_path = Path(file_path)
    if not input_path.exists():
        print(f" File not found: {input_path}")
        return

    is_audio = input_path.suffix.lower() in AUDIO_EXTS
    stem = input_path.stem
    output_base_dir = Path(output_base_dir) if output_base_dir else input_path.parent

    folder_name = f"{stem}_chunks_{int(segment_duration)}s"
    save_dir = output_base_dir / folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    try:
        if is_audio:
            clip = AudioFileClip(str(input_path))
        else:
            clip = VideoFileClip(str(input_path))
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    # --- THE CRITICAL FIX: Pretend the file is 5s shorter ---
    duration = clip.duration - trim_end_sec 
    
    if drop_last:
        num_chunks = math.floor(duration / segment_duration)
        drop_msg = "(Dropping last partial chunk)"
    else:
        num_chunks = math.ceil(duration / segment_duration)
        drop_msg = "(Keeping last partial chunk)"
        
    pad_len = max(3, len(str(num_chunks)))

    print(f"--- Processing: {input_path.name} ---")
    print(f"Type: {'Audio' if is_audio else 'Video'}")
    print(f"Orig Duration: {clip.duration:.2f}s | Trimmed Duration: {duration:.2f}s | Count: {num_chunks}")

    for i in tqdm(range(num_chunks), desc=f"Segmenting {stem}"):
        start_t = i * segment_duration
        end_t = min((i + 1) * segment_duration, duration)
        
        if is_audio:
            chunk_name = f"{stem}_part_{i+1:0{pad_len}d}.wav"
        else:
            chunk_name = f"{stem}_part_{i+1:0{pad_len}d}{input_path.suffix}"
            
        chunk_path = save_dir / chunk_name
        
        try:
            subclip = clip.subclipped(start_t, end_t)
            if is_audio:
                subclip.write_audiofile(str(chunk_path), codec='pcm_s16le', logger=None)
            else:
                subclip.write_videofile(
                    str(chunk_path), codec="libx264", audio=False, preset="ultrafast", threads=4, logger=None
                )
        except Exception as e:
            print(f"Failed on chunk {i+1}: {e}")
    
    clip.close()
    print(f"Finished. Saved to: {save_dir}\n")


# --- RUN EXECUTION ---
SEGMENT_SEC_LIST = [1.0, 2.0, 5.0, 10.0]
INPUT_ROOT = Path(r"/home/amin/Research/Representation/Movie/data/Setareh/runs") 

run_files = sorted(INPUT_ROOT.glob("*.mp4"))

for target_video in run_files:
    print(f"\n========== Processing Full Run: {target_video.name} ==========")
    
    target_audio = INPUT_ROOT / f"{target_video.stem}_audio.wav"
    
    # Extract full audio first if needed
    if not target_audio.exists():
        print(f"Extracting full audio from {target_video.name}...")
        try:
            clip = VideoFileClip(str(target_video))
            if clip.audio is not None:
                clip.audio.write_audiofile(str(target_audio), codec='pcm_s16le', logger=None)
            clip.close()
        except Exception as e:
            print(f"Error extracting audio from {target_video.name}: {e}")
            continue

    # Segment using the new trimmed function
    for segment_duration in SEGMENT_SEC_LIST:
        print(f"\n--- Segmenting {segment_duration}s chunks ---")
        
        segment_media_trimmed(
            file_path=target_video, 
            segment_duration=segment_duration,
            trim_end_sec=5.0
        )
        
        if target_audio.exists():
            segment_media_trimmed(
                file_path=target_audio, 
                segment_duration=segment_duration,
                trim_end_sec=5.0
            )


========== Processing Full Run: 7T_MOVIE1_CC1_v2.mp4 ==========

--- Segmenting 1.0s chunks ---
--- Processing: 7T_MOVIE1_CC1_v2.mp4 ---
Type: Video
Orig Duration: 921.00s | Trimmed Duration: 916.00s | Count: 916


Segmenting 7T_MOVIE1_CC1_v2:   0%|          | 0/916 [00:00<?, ?it/s]

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/runs/7T_MOVIE1_CC1_v2_chunks_1s

--- Processing: 7T_MOVIE1_CC1_v2_audio.wav ---
Type: Audio
Orig Duration: 921.00s | Trimmed Duration: 916.00s | Count: 916


Segmenting 7T_MOVIE1_CC1_v2_audio:   0%|          | 0/916 [00:00<?, ?it/s]

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/runs/7T_MOVIE1_CC1_v2_audio_chunks_1s


--- Segmenting 2.0s chunks ---
--- Processing: 7T_MOVIE1_CC1_v2.mp4 ---
Type: Video
Orig Duration: 921.00s | Trimmed Duration: 916.00s | Count: 458


...

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/runs/7T_MOVIE4_HO2_v2_audio_chunks_5s


--- Segmenting 10.0s chunks ---
--- Processing: 7T_MOVIE4_HO2_v2.mp4 ---
Type: Video
Orig Duration: 901.00s | Trimmed Duration: 896.00s | Count: 89


Segmenting 7T_MOVIE4_HO2_v2:   0%|          | 0/89 [00:00<?, ?it/s]

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/runs/7T_MOVIE4_HO2_v2_chunks_10s

--- Processing: 7T_MOVIE4_HO2_v2_audio.wav ---
Type: Audio
Orig Duration: 901.00s | Trimmed Duration: 896.00s | Count: 89


Segmenting 7T_MOVIE4_HO2_v2_audio:   0%|          | 0/89 [00:00<?, ?it/s]

Finished. Saved to: /home/amin/Research/Representation/Movie/data/Setareh/runs/7T_MOVIE4_HO2_v2_audio_chunks_10s

